In [13]:
%pip install -q \
    fpdf \
    Pillow


[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
from fpdf import FPDF
import os
from PIL import Image
import io

class PDF(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 12)
        self.cell(0, 10, 'Receipts', 0, 1, 'C')

    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.cell(0, 10, f'Page {self.page_no()}', 0, 0, 'C')

def optimize_image(image_path, max_width=800, quality=85, dpi=150):
    """Optimize an image by resizing and compressing it."""
    img = Image.open(image_path)
    
    # Calculate new dimensions while maintaining aspect ratio
    width_percent = max_width / float(img.size[0])
    new_height = int(float(img.size[1]) * width_percent)
    
    # Resize the image
    if img.size[0] > max_width:
        img = img.resize((max_width, new_height), Image.LANCZOS)
    
    # Convert to RGB if it's RGBA (removes alpha channel)
    if img.mode == 'RGBA':
        img = img.convert('RGB')
    
    # Save to memory buffer with compression
    buffer = io.BytesIO()
    img.save(buffer, format='JPEG', quality=quality, dpi=(dpi, dpi))
    buffer.seek(0)
    
    return buffer

def create_receipts_pdf(directory, output_file, max_width=600, quality=75, dpi=100):
    pdf = PDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.set_compression(True)  # Enable PDF compression
    
    # Create a temporary directory for optimized images
    import tempfile
    temp_dir = tempfile.mkdtemp()
    
    try:
        for filename in os.listdir(directory):
            if filename.endswith(('.png', '.jpg', '.jpeg')):
                pdf.add_page()
                image_path = os.path.join(directory, filename)
                
                # Optimize the image
                img_buffer = optimize_image(image_path, max_width, quality, dpi)
                
                # Save buffer to a temporary file
                temp_image_path = os.path.join(temp_dir, f"temp_{filename}")
                with open(temp_image_path, 'wb') as f:
                    f.write(img_buffer.getvalue())
                
                # Add image from temp file
                pdf.image(temp_image_path, x=10, w=0, h=pdf.h-50)
        
        pdf.output(output_file)
    
    finally:
        # Clean up temporary files
        import shutil
        shutil.rmtree(temp_dir)

# Specify the directory and output file
uploads_directory = '/home/alex/src/github.com/receipts/frontend/public/uploads'
output_pdf_file = '/home/alex/src/github.com/receipts/receipts.pdf'

# Create the PDF with optimized images
create_receipts_pdf(uploads_directory, output_pdf_file)